# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2. Datos

In [2]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [3]:
df.head()
df.info()
df.describe()
df.isnull().sum()
df.nunique()


<class 'pandas.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    str    
 2   Product           912 non-null    str    
 3   TypeName          912 non-null    str    
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    str    
 6   Cpu               912 non-null    str    
 7   Ram               912 non-null    str    
 8   Memory            912 non-null    str    
 9   Gpu               912 non-null    str    
 10  OpSys             912 non-null    str    
 11  Weight            912 non-null    str    
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), str(10)
memory usage: 92.8 KB


laptop_ID           912
Company              19
Product             480
TypeName              6
Inches               17
ScreenResolution     36
Cpu                 107
Ram                   9
Memory               37
Gpu                  93
OpSys                 9
Weight              165
Price_in_euros      603
dtype: int64

### 2.2 Definir X e y


In [4]:
def feature_engineer(data):
    """Convierte las columnas 'sucias' del dataset en features numéricas/categóricas limpias."""
    data = data.copy()

    # Ram: '8GB' -> 8
    data['Ram'] = data['Ram'].astype(str).str.replace('GB', '', regex=False).astype(int)

    # Weight: '1.86kg' -> 1.86
    data['Weight'] = data['Weight'].astype(str).str.replace('kg', '', regex=False).astype(float)

    # ScreenResolution: touchscreen, IPS, resolución y densidad de píxeles (PPI)
    data['Touchscreen'] = data['ScreenResolution'].str.contains('Touchscreen').astype(int)
    data['IPS'] = data['ScreenResolution'].str.contains('IPS').astype(int)
    res = data['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
    data['ScreenWidth'] = res[0].astype(int)
    data['ScreenHeight'] = res[1].astype(int)
    data['PPI'] = ((data['ScreenWidth']**2 + data['ScreenHeight']**2)**0.5) / data['Inches']

    # Cpu: marca/gama y velocidad en GHz
    data['CpuBrand'] = data['Cpu'].apply(lambda x: ' '.join(x.split()[:2]) if 'Intel' in x else x.split()[0])
    data['CpuSpeedGHz'] = data['Cpu'].str.extract(r'([\d\.]+)GHz').astype(float)

    # Memory: puede tener varias unidades combinadas ("256GB SSD +  1TB HDD")
    def parse_memory(mem):
        mem = mem.replace('GB', '').replace('TB', '000')
        ssd = hdd = flash = hybrid = 0.0
        for part in mem.split('+'):
            part = part.strip()
            num = float(''.join(c for c in part if (c.isdigit() or c == '.')))
            if 'SSD' in part:
                ssd += num
            elif 'HDD' in part:
                hdd += num
            elif 'Flash' in part:
                flash += num
            elif 'Hybrid' in part:
                hybrid += num
        return pd.Series([ssd, hdd, flash, hybrid])

    data[['SSD_GB', 'HDD_GB', 'Flash_GB', 'Hybrid_GB']] = data['Memory'].apply(parse_memory)

    # Gpu: solo la marca (Intel / Nvidia / AMD / ARM)
    data['GpuBrand'] = data['Gpu'].apply(lambda x: x.split()[0])

    # Columnas ya no necesarias: son texto libre o quedaron reemplazadas por las nuevas features
    data = data.drop(columns=['ScreenResolution', 'Cpu', 'Memory', 'Gpu', 'Product'])

    return data


df_fe = feature_engineer(df)

X = df_fe.drop(columns=['laptop_ID', 'Price_in_euros'])
y = df_fe['Price_in_euros']

cat_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'str']).columns.tolist()

print('Columnas categóricas:', cat_cols)
print('Columnas numéricas:', num_cols)
X.head()

Columnas categóricas: ['Company', 'TypeName', 'OpSys', 'CpuBrand', 'GpuBrand']
Columnas numéricas: ['Inches', 'Ram', 'Weight', 'Touchscreen', 'IPS', 'ScreenWidth', 'ScreenHeight', 'PPI', 'CpuSpeedGHz', 'SSD_GB', 'HDD_GB', 'Flash_GB', 'Hybrid_GB']


,Company,TypeName,Inches,Ram,OpSys,Weight,Touchscreen,IPS,ScreenWidth,ScreenHeight,PPI,CpuBrand,CpuSpeedGHz,SSD_GB,HDD_GB,Flash_GB,Hybrid_GB,GpuBrand
0,HP,Notebook,15.6,8,Windows 10,1.86,0,0,1920,1080,141.211998,Intel Core,2.0,256.0,0.0,0.0,0.0,Intel
1,Dell,Gaming,15.6,16,Windows 10,2.59,0,0,1920,1080,141.211998,Intel Core,2.6,0.0,1000.0,0.0,0.0,Nvidia
2,HP,Notebook,15.6,8,Windows 10,2.04,0,0,1920,1080,141.211998,Intel Core,2.7,0.0,1000.0,0.0,0.0,Nvidia
3,Apple,Ultrabook,13.3,8,macOS,1.34,0,0,1440,900,127.677940,Intel Core,1.8,0.0,0.0,128.0,0.0,Intel
4,Dell,Notebook,15.6,4,Linux,2.25,0,0,1920,1080,141.211998,Intel Core,2.0,0.0,1000.0,0.0,0.0,AMD


### 2.3 Dividir en train y test

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train:', X_train.shape, ' Test:', X_test.shape)

Train: (729, 18)  Test: (183, 18)


## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [6]:
# One-Hot Encoding para las categóricas (fit SOLO sobre X_train)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[cat_cols])

# Escalado para las numéricas (fit SOLO sobre X_train) — útil sobre todo para el SVR
scaler = StandardScaler()
scaler.fit(X_train[num_cols])


def preprocess(X_part):
    """Aplica SIEMPRE .transform() (nunca .fit) para evitar data leakage."""
    cat_arr = ohe.transform(X_part[cat_cols])
    cat_df = pd.DataFrame(
        cat_arr, columns=ohe.get_feature_names_out(cat_cols), index=X_part.index
    )

    num_arr = scaler.transform(X_part[num_cols])
    num_df = pd.DataFrame(num_arr, columns=num_cols, index=X_part.index)

    return pd.concat([num_df, cat_df], axis=1)


X_train_proc = preprocess(X_train)
X_test_proc = preprocess(X_test)

X_train_proc.shape, X_test_proc.shape

((729, 54), (183, 54))

## 4. Modelado

### 4.1 Entrenamiento

In [7]:
rf_model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf_model.fit(X_train_proc, y_train)

svr_model = SVR(kernel='rbf', C=1000, epsilon=10)
svr_model.fit(X_train_proc, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1000
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",10
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [8]:
for name, model in [('RandomForest', rf_model), ('SVR', svr_model)]:
    preds = model.predict(X_test_proc)
    rmse = root_mean_squared_error(y_test, preds)
    print(f'{name:12s} -> RMSE: {rmse:.2f} €')

RandomForest -> RMSE: 373.55 €
SVR          -> RMSE: 363.28 €


### 4.3 Optimización (up to you 🫰🏻)

In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train_proc, y_train)

print('Mejores parámetros:', grid_search.best_params_)
print('Mejor RMSE (CV):', -grid_search.best_score_)

best_model = grid_search.best_estimator_
preds = best_model.predict(X_test_proc)
print('RMSE en test local:', root_mean_squared_error(y_test, preds))

Mejores parámetros: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 600}
Mejor RMSE (CV): 292.71566644763095
RMSE en test local: 375.1532982245277


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [10]:
# Reajustamos encoder y scaler con TODO train.csv...
ohe_final = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_final.fit(X[cat_cols])

scaler_final = StandardScaler()
scaler_final.fit(X[num_cols])


def preprocess_final(X_part):
    cat_arr = ohe_final.transform(X_part[cat_cols])
    cat_df = pd.DataFrame(
        cat_arr, columns=ohe_final.get_feature_names_out(cat_cols), index=X_part.index
    )
    num_arr = scaler_final.transform(X_part[num_cols])
    num_df = pd.DataFrame(num_arr, columns=num_cols, index=X_part.index)
    return pd.concat([num_df, cat_df], axis=1)


X_full_proc = preprocess_final(X)

# ...y reentrenamos el modelo ganador (RandomForest optimizado) con el 100% de los datos
final_model = RandomForestRegressor(**grid_search.best_params_, random_state=42, n_jobs=-1)
final_model.fit(X_full_proc, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",600
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [11]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [12]:
X_pred_fe = feature_engineer(X_pred)

# Nos aseguramos de no perder ninguna fila (392 en test.csv)
assert len(X_pred_fe) == len(X_pred)

X_pred_features = X_pred_fe.drop(columns=['laptop_ID'])

# ¡SOLO transform! El encoder y el scaler ya están "fit" con train.csv
X_pred_proc = preprocess_final(X_pred_features)

# Nos aseguramos de que las columnas coincidan exactamente con las de entrenamiento
X_pred_proc = X_pred_proc.reindex(columns=X_full_proc.columns, fill_value=0)

X_pred_proc.shape

(391, 56)

## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [13]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

In [14]:
predicted_prices = final_model.predict(X_pred_proc)

submission = pd.DataFrame({
    'laptop_ID': X_pred_fe['laptop_ID'],
    'Price_in_euros': predicted_prices
})

submission.head()

,laptop_ID,Price_in_euros
0,209,1503.641283
1,1281,289.281508
2,1168,406.003183
3,1231,1037.884400
4,1020,1134.863683


### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [15]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [16]:
checker(submission, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260914_111135.csv'. ¡A Kaggle!
